# StashFace — مطابقة على Kaggle (2x T4 GPU)

**قبل ما تشغل أي خلية:** من إعدادات الـnotebook (يمين الشاشة) اختار Accelerator = **GPU T4 x2**.

شغّل الخلايا بالترتيب من فوق لتحت.

In [ ]:
# ============================================================
# خلية الإعداد — عدّل القيم دي بس، والباقي متلمسوش
# ============================================================

# رابط الـGitHub repo بتاع المشروع
GITHUB_REPO_URL = "https://github.com/kareemkamal10/stashface_pipeline"

# الـtoken بتاع حسابك على Hugging Face (لازم يكون عنده صلاحية Write)
HF_TOKEN = " "

# الـdataset اللي فيه ملفات الأداء، وهيتحفظ فيه كل تقارير الفحص في الآخر
HF_DATASET_ID = "abdelwahabnabil500/datafile"

# اسم الملفين جوه الـdataset (نفس الاسمين المحليين بالظبط)
HF_INPUT_WITH_TPDB = "performers_with_tpdb.json"
HF_INPUT_WITHOUT_TPDB = "performers_without_tpdb.json"

In [2]:
# ============================================================
# تحميل الكود وتثبيت المكتبات — مفيش حاجة تتعدل هنا
# ============================================================
import os

# منع hf CLI من سؤال "تحدّث دلوقتي؟" اللي بيعلّق جوه notebook (مفيش حد يرد عليه)
os.environ["HF_HUB_DISABLE_UPDATE_CHECK"] = "1"

# 1) تحميل الكود من الـGitHub repo
!git clone {GITHUB_REPO_URL} /kaggle/working/stashface_pipeline
%cd /kaggle/working/stashface_pipeline

# 2) تثبيت المكتبات المطلوبة
!pip install -q -r requirements.txt
!pip install -q -U "huggingface_hub[cli]>=1.13.0"

# 3) استبدال onnxruntime بنسخة الـGPU — عشان يستخدم الـT4 فعليًا مش الـCPU
!pip uninstall -y -q onnxruntime
!pip install -q "onnxruntime-gpu==1.26.0"  # pinned: onnxruntime-gpu>=1.27 defaults to CUDA 13, Kaggle T4 images still run CUDA 12.x

# 4) تسجيل الدخول لـHugging Face بالـtoken بتاعك
from huggingface_hub import login
login(token=HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN

Cloning into '/kaggle/working/stashface_pipeline'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 100 (delta 29), reused 38 (delta 16), pack-reused 47 (from 1)
Receiving objects: 100% (100/100), 96.89 MiB | 24.64 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/kaggle/working/stashface_pipeline
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 70.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 762.2/762.2 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 78.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 66.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 36.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 77.4 MB/s eta 0:00:

In [ ]:
# ============================================================
# تحميل بيانات المشروع (الموديل) + ملفي الأداء
# ============================================================

# 5) تحميل بيانات المشروع (adaface model) من الـbucket
!python setup.py --skip-install

# 6) تحميل الملفين من الـdataset بتاعك
from huggingface_hub import hf_hub_download
import shutil

for remote_name, local_name in [
    (HF_INPUT_WITH_TPDB, "performers_with_tpdb.json"),
    (HF_INPUT_WITHOUT_TPDB, "performers_without_tpdb.json"),
]:
    local_path = hf_hub_download(
        repo_id=HF_DATASET_ID,
        repo_type="dataset",
        filename=remote_name,
        token=HF_TOKEN,
    )
    shutil.copy(local_path, local_name)
    print("تم تحميل:", local_name)

## التشغيل الكامل على كل البيانات

دي هتاخد وقت طويل (ساعات، حسب حجم البيانات). لو الجلسة اتقفلت أو حصل أي كراش، رجّع شغّل نفس الخلية تاني — هيكمل من حيث ما وقف من غير ما يعيد اللي خلص منه.

بيني الـDB من الملفين مع بعض، يفحص كل عنصر (تحميل + كشف وش)، يعمل استعلام top-10 لكل عنصر مقبول، ويقسم النتايج لـ"تطابقات مؤكدة" و"محتاجة مراجعة بشرية". في الآخر بيرفع كل التقارير + ملف الـDB تلقائيًا لمجلد `reports/` جوا نفس الـdataset — مفيش خلية رفع منفصلة بعد كده.

In [ ]:
!python dedupe_pipeline.py --device-id 0 --hf-dataset-id {HF_DATASET_ID} --hf-token {HF_TOKEN}

## ملخص النتايج

السكريبت رفع كل حاجة أوتوماتيك في الخلية اللي فاتت (مجلد `reports/` جوا الـdataset). الخلية دي بس بتوريك ملخص سريع محليًا قبل ما تقفل الـnotebook.

In [ ]:
import json, os

for fname in [
    "dedupe_reports/face_db.npz",
    "dedupe_reports/download_failed.json",
    "dedupe_reports/no_face_detected.json",
    "dedupe_reports/multiple_faces_detected.json",
    "dedupe_reports/confirmed_duplicates.json",
    "dedupe_reports/needs_human_review.json",
]:
    if not os.path.exists(fname):
        print(f"{fname}: مش موجود")
        continue
    if fname.endswith(".json"):
        n = len(json.load(open(fname, encoding="utf-8")))
        print(f"{fname}: {n} سجل")
    else:
        size_mb = os.path.getsize(fname) / (1024 * 1024)
        print(f"{fname}: {size_mb:.1f} MB")